In [1]:
# multi class classification of FGL sources using tree categories
import numpy as np
import pandas as pd
from matplotlib import pyplot
#from scipy.stats import binom
print(np.__version__)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.calibration import calibration_curve

import time
import os
import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
from fermi_multiclass_ps.classification import (
    _fit_classifier, accumulate_classifier_predictions, build_classification_targets,
    mean_and_standard_deviation,
)
import plot_util as putil

1.24.3


In [2]:
if 1:
    import importlib
    importlib.reload(util);


In [3]:
# select the catalog and the case
save_all_preds = 1
cat_name = '4FGL-DR4' # '4FGL-DR3'
if cat_name == '4FGL-DR4':
    # cases for GU & KDE papers
    baseline_case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF' # 49 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu' # 51 train / test splits
    case = '4FGL-DR4_GMM_RF_no_coord' # 49 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp_w' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_w' # 49 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp_w' # 48 train / test splits
    
    #baseline_case = '4FGL-DR4_2class_RF' # '4FGL-DR4_GMM_RF', '4FGL-DR4_2class_RF'
    #case = '4FGL-DR4_2class_pulsars_AGNs'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 47 train/test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 47 train/test splits
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord' # 51 train/test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_w' # 51 train/test splits
if cat_name == '4FGL-DR3':
    #baseline_case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp' # 57 splits
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp_nmin50' # 57 splits
    baseline_case = '4FGL-DR3_3class_RF'
    case = '4FGL-DR3_3class_RF'
    #case = '4FGL-DR3_3class_spec_features_RF'

baseline = (case == baseline_case)
root_folder = '../'
config_folder = root_folder + 'config/'

# generic inputs
cdict = util.get_config_dict(#cat_name=cat_name, 
                             baseline_case=baseline_case, 
                             #baseline_fn=baseline_fn,
                             #cases_fn=cases_fn,
                             case=case, 
                             folder=config_folder)


cdict['root_folder'] = root_folder
folder_in = cdict['root_folder'] + cdict.get('folder_in', 'data_in/')
folder_out = cdict['root_folder'] + cdict.get('folder_out', 'data_out/')
folder = folder_out
Eind = int(cdict.get('Eind', 1000))
class_name = cdict.get('class_name', 'CLASS1')

assoc_sources = cdict.get('assoc_sources', 'all_assoc')
sep_model = cdict.get('sep_model', 'GMM')
#sep_model = 'RF' # 'GMM', 'RF'
nmin = cdict.get('nmin')
if nmin is not None: 
    nmin = int(nmin)
max_depth = int(cdict.get('max_depth', 5))
feature_mode = cdict.get('features', 'all_features')

alg = cdict.get('class_alg', 'RF')
train_weighting = cdict.get('train_weighting')
test_weighting = cdict.get('test_weighting')
weighted = (train_weighting is not None) or (test_weighting is not None)
#oversample_factor = 1 # used for 'resampled' training and testing
max_sample = int(cdict.get('max_sample', 7))
test_frac = float(cdict.get('test_frac', 0.3))
nav = int(cdict.get('nav', 10))


tree_alg = cdict.get('sep_model', 'GMM')
#tree_alg = 'RF' # 'stat', 'GMM', 'RF', '4class', '2class', 'RF_aakash'


# input data
tree_cat_fn = util.get_tree_cat_fn(cdict)
tree_cat_dict = dio3.loaddict(tree_cat_fn)
if util.exist_internal(tree_cat_dict):
    util.remove_internal(tree_cat_dict)
cat_tree = list(tree_cat_dict.keys())
nclasses = len(tree_cat_dict)
max_tree_cat_dict = util.tree2max_tree(tree_cat_dict)
dec_tree_cat_dict = util.max_tree2dec_tree(max_tree_cat_dict)
nodes = [str(key) for key in dec_tree_cat_dict.keys()]


xdf_fn = util.get_xdf_fn(cdict)
xdf = pd.read_csv(xdf_fn, index_col=0)
ns = len(xdf)

all_features = [name for name in xdf.columns if name.find('CLASS') == -1]
if feature_mode == 'all_features':
    features = all_features
elif feature_mode == 'no_coord_features':
    features = [feat for feat in all_features if feat.find('GL') == -1]
elif feature_mode == 'spec_features':
    # should be equivalent to Toby's features
    features = ['LP_index1000MeV', 'LP_beta', 'log10(Energy_Flux100)', 'log10(Variability_Index)']

nsrc_dict = util.get_nsrc_dict(xdf, class_column=class_name, classes='associated')
assoc_classes = list(nsrc_dict.keys())

# algorithm description string
pars_dict = util.pars_dict
pars_dict['NN']['nfeatures'] = len(features)
pars_dict['NN']['classes'] = nodes
pars_dict['NN']['epochs'] = cdict.get('epochs', 500)
alg_descr = util.get_alg_descr(cdict, pars_dict=pars_dict)

weights_fn = util.get_weights_fn(cdict)
if os.path.isfile(weights_fn):
    weights_df = pd.read_csv(weights_fn, index_col=0)
    sample_ratio_0 = util.get_sample_ratio(weights_df['unas_pdf'], weights_df['assoc_pdf'], binary=False, 
                                           max_sample=max_sample)
else:
    sample_ratio_0 = 1.

if weighted:
    weight_str = '_weighted'
    sample_ratio = sample_ratio_0
else:
    sample_ratio = None
    weight_str = ''
    
folder_out = '../results/%s_probabilistic_catalogs/' % util.get_cat_descr(cdict)
if not os.path.isdir(folder_out):
    os.mkdir(folder_out)
fn_out = folder_out + util.get_cat_descr(cdict) + '_prob_cat.csv'

#class_def_fn = folder_out + '4FGL-DR3_%iclass%s_groups_def.csv' % (nclasses, alg_descr)

#algs = ['RF', 'SVC', 'NN']
algs = ['RF', 'NN']
#algs = ['RF',]
print(nclasses)
print(tree_cat_fn)
print(fn_out)

6
../data_out//tree_cat_4FGL-DR4_sep_model_GMM_nmin_100_all_assoc_no_coord_features.yaml
../results/4FGL-DR4_v34_6classes_GMM_no_coord_features_probabilistic_catalogs/4FGL-DR4_v34_6classes_GMM_no_coord_features_prob_cat.csv


In [4]:
all_features

['sin(GLAT)',
 'cos(GLON)',
 'sin(GLON)',
 'log10(Energy_Flux100)',
 'log10(Unc_Energy_Flux100)',
 'log10(Signif_Avg)',
 'LP_index1000MeV',
 'LP_beta',
 'LP_SigCurv',
 'log10(Variability_Index)']

In [5]:
dec_tree_cat_dict

{1: ['nov', 'spp'],
 2: ['fsrq', 'nlsy1'],
 3: ['snr', 'hmb', 'pwn', 'psr', 'gc'],
 4: ['msp', 'lmb', 'glc', 'gal', 'sfr', 'bin'],
 5: ['sey', 'bcu', 'sbg', 'agn', 'rdg'],
 6: ['bll', 'ssrq', 'css']}

In [6]:
X, X_unas, y, clf_mask = build_classification_targets(
    xdf, dec_tree_cat_dict, features=features, class_column=class_name
)


classif_columns = ['%s_%s' % (node, alg) for node in nodes for alg in algs]
print(classif_columns)



['1_RF', '1_NN', '2_RF', '2_NN', '3_RF', '3_NN', '4_RF', '4_NN', '5_RF', '5_NN', '6_RF', '6_NN']


In [7]:
add_sample_weights_only = False
if add_sample_weights_only:
    cat_df = pd.read_csv(fn_out, index_col=0)
    cat_df['sample_weight'] = sample_ratio_0
    cat_df['sample_weight'][~clf_mask] = 1.
    print('Save the prob catalog to file:')
    print(fn_out)
    print(cat_df.columns)
    cat_df.to_csv(fn_out)
    print(cat_df[['CLASS1', 'sample_weight']])

In [8]:
#print(cat_df[['CLASS1', 'sample_weight']][:40])
#print(np.max(cat_df['sample_weight']))

In [9]:
y_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
y2_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
nn_df = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0)


In [10]:
low_lim = 5
if save_all_preds:
    all_folder_out = fn_out.replace('.csv', '_all_pred/')
    if not os.path.isdir(all_folder_out):
        os.mkdir(all_folder_out)

for alg in algs:
    j = -1
    print(alg)
    alg_clms = [clm for clm in classif_columns if clm.find(alg) > -1]
    while np.min(nn_df[alg_clms].values) < low_lim:
        if alg.startswith('NN'):
            time0 = time.time()
        if save_all_preds:
            classif_columns_loc = ['%s_%s' % (node, alg) for node in nodes]
            y_pred_loc = pd.DataFrame(index=xdf.index, columns=classif_columns_loc, data=0.)
        j += 1
        print(j, np.min(nn_df[alg_clms].values))
        clf = util.get_classifier(alg, pars_dict[alg])
        if 1:
            clf_inds = np.arange(len(xdf))[clf_mask]
            rstate = np.random.RandomState(seed=j)
            test_inds = rstate.choice(clf_inds, size=int(len(clf_inds) * test_frac), replace=False, p=None)
            train_inds = [i for i in clf_inds if i not in test_inds]
            
            # add weights to training samples
            if weighted:
                sample_ratio_train = sample_ratio.iloc[train_inds]
            else:
                sample_ratio_train = None
            # add weights to testing samples
            #pred_dict_all[j]['test_weights'] = sample_ratio.iloc[test_inds]

            X_train = xdf[features].iloc[train_inds]
            X_test = xdf[features].iloc[test_inds]
            y_train = ys[train_inds]
            y_test = ys[test_inds]
            
            #clf.fit(X_train, y_train, sample_weight=sample_ratio_train)
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_frac, 
                                                                random_state=j)
        _fit_classifier(clf, X_train, y_train, sample_ratio_train)

        y_pred_loc = accumulate_classifier_predictions(
            clf, X_test, X_unas, nodes=nodes, algorithm=alg,
            prediction_sum=y_pred, prediction_square_sum=y2_pred,
            prediction_count=nn_df, save_all=save_all_preds,
        )
        if save_all_preds:
            fn = all_folder_out + '%s_%i.npy' % (alg, j)
            np.save(fn, y_pred_loc.to_numpy())


        if alg.startswith('NN'):
            print('Step time: %.2f s' %  (time.time() - time0))


RF
0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 0
13 0
14 0
15 0
16 0
17 0
18 0
19 0
20 0
21 0
22 0
23 1
24 1
25 1
26 1
27 1
28 1
29 1
30 2
31 2
32 2
33 2
34 2
35 2
36 2
37 2
38 2
39 2
40 3
41 3
42 3
43 4
44 4
45 4
46 4
47 4
48 4
NN
0 0


2024-08-19 15:31:50.874762: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-19 15:32:03.146648: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Step time: 19.74 s
1 0
Step time: 7.04 s
2 0
Step time: 7.13 s
3 0
Step time: 7.15 s
4 0
Step time: 7.10 s
5 0
Step time: 7.31 s
6 0
Step time: 7.39 s
7 0
Step time: 7.60 s
8 0
Step time: 7.43 s
9 0
Step time: 7.25 s
10 0
Step time: 7.16 s
11 0
Step time: 7.22 s
12 0
Step time: 7.60 s
13 0
Step time: 7.16 s
14 0
Step time: 7.21 s
15 0
Step time: 7.20 s
16 0
Step time: 7.20 s
17 0
Step time: 7.25 s
18 0
Step time: 7.22 s
19 0
Step time: 7.34 s
20 0
Step time: 7.65 s
21 0
Step time: 7.21 s
22 0
Step time: 7.25 s
23 1
Step time: 7.22 s
24 1
Step time: 7.20 s
25 1
Step time: 7.42 s
26 1
Step time: 7.52 s
27 1
Step time: 7.57 s
28 1
Step time: 7.19 s
29 1
Step time: 7.21 s
30 2
Step time: 7.22 s
31 2
Step time: 7.26 s
32 2
Step time: 7.18 s
33 2
Step time: 7.23 s
34 2
Step time: 7.20 s
35 2
Step time: 7.25 s
36 2
Step time: 7.29 s
37 2
Step time: 7.22 s
38 2
Step time: 7.37 s
39 2
Step time: 7.18 s
40 3
Step time: 7.17 s
41 3
Step time: 7.18 s
42 3
Step time: 7.29 s
43 4
Step time: 7.27 s
4

In [11]:
y_pred_norm, y_std = mean_and_standard_deviation(y_pred, y2_pred, nn_df)

In [12]:
#sdf.index = range(1, len(sdf) + 1)

In [13]:
# output the catalog
df_out = xdf.copy()
for clm in classif_columns:
    df_out[clm] = y_pred_norm[clm]
    df_out[clm + '_std'] = y_std[clm]
df_out['sample_weight'] = sample_ratio_0
df_out['sample_weight'][~clf_mask] = 1. # put 1 for unassociated sources

print('Save the prob catalog to file:')
print(fn_out)
df_out.to_csv(fn_out)

if save_all_preds:
    fn = all_folder_out + 'ntest.npy'
    print(fn)
    nns = np.array(nn_df[nn_df.columns[0]])
    print(nns[:10])
    np.save(fn, nns)


Save the prob catalog to file:
../results/4FGL-DR4_v34_6classes_GMM_no_coord_features_probabilistic_catalogs/4FGL-DR4_v34_6classes_GMM_no_coord_features_prob_cat.csv
../results/4FGL-DR4_v34_6classes_GMM_no_coord_features_probabilistic_catalogs/4FGL-DR4_v34_6classes_GMM_no_coord_features_prob_cat_all_pred/ntest.npy
[49 49 49 17 13 15 19 49 15 21]


/var/folders/mb/yxqk032n4lvgsvnl3lqzbwvr0000gn/T/ipykernel_34925/2302742597.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_out['sample_weight'][~clf_mask] = 1. # put 1 for unassociated sources


In [14]:
df_out.columns

Index(['sin(GLAT)', 'cos(GLON)', 'sin(GLON)', 'log10(Energy_Flux100)',
       'log10(Unc_Energy_Flux100)', 'log10(Signif_Avg)', 'LP_index1000MeV',
       'LP_beta', 'LP_SigCurv', 'log10(Variability_Index)', 'CLASS1', '1_RF',
       '1_RF_std', '1_NN', '1_NN_std', '2_RF', '2_RF_std', '2_NN', '2_NN_std',
       '3_RF', '3_RF_std', '3_NN', '3_NN_std', '4_RF', '4_RF_std', '4_NN',
       '4_NN_std', '5_RF', '5_RF_std', '5_NN', '5_NN_std', '6_RF', '6_RF_std',
       '6_NN', '6_NN_std', 'sample_weight'],
      dtype='object')

In [15]:
sum(df_out['CLASS1'] == 'unk')

147

In [16]:
sample_ratio_train

In [17]:
dec_tree_cat_dict

{1: ['nov', 'spp'],
 2: ['fsrq', 'nlsy1'],
 3: ['snr', 'hmb', 'pwn', 'psr', 'gc'],
 4: ['msp', 'lmb', 'glc', 'gal', 'sfr', 'bin'],
 5: ['sey', 'bcu', 'sbg', 'agn', 'rdg'],
 6: ['bll', 'ssrq', 'css']}